In [1]:
import collections
from glob import glob
import joblib
import pandas as pd
from matplotlib import pyplot as plt
from torch.utils.tensorboard import SummaryWriter
import os
import numpy as np
import torch.nn as nn
import torch.utils.data
from torchvision import transforms
from PIL import ImageFile
import torchvision.models as models
from PIL import Image, ImageFile
import scanpy as sc

ImageFile.LOAD_TRUNCATED_IMAGES = True


class PTC_cell_2x(torch.utils.data.Dataset):
    def __init__(self, root, slide='11_breast_cancer3',transform=None, stain_norm=False):
        super(PTC_cell_2x, self).__init__()
        self.root = root
        self.slide = slide
        self.transform = transform
        self.stain_norm = stain_norm

        patch_path = os.path.join(root, slide, '2x_patches')
        patch = os.listdir(patch_path)
        patch_list = [x.split('.j')[0] for x in patch]

        cell_label = pd.read_csv(os.path.join(root, slide, 'cell_ratio.csv'), index_col=0)
        # cell_label = cell_label.apply(lambda x: x*100.0)
        
        gene_label = pd.read_csv(os.path.join(root, slide, 'high_250_stdata.csv'), index_col=0)
        label_df = pd.merge(gene_label, cell_label, left_index=True, right_index=True)

        # label_index_set = set(label_df.index)
        # patch_index_set = set(patch_list)
        # and_set = label_index_set & patch_index_set
        # patch_list = list(and_set)
        # self.label_df = label_df.loc[patch_list]
        
        self.label_df = label_df
        self.patch = patch_list


    def __getitem__(self, index):
        patch_id = self.patch[index]
        patch_path = os.path.join(self.root, self.slide, '2x_patches', patch_id)
        patch = Image.open(patch_path+'.jpg').convert('RGB')
        data = transforms.Resize((224, 224))(patch)
        if self.transform is not None:
            data = self.transform(data)

        if patch_id in self.label_df.index:
            label = self.label_df.loc[patch_id].values
        else:
            label = np.full((self.label_df.iloc[0].values.shape), -1.0)
        label = torch.Tensor(label)

        return patch_id, data, label

    def __len__(self):
        return len(self.patch)


class fully_connected(nn.Module):
    def __init__(self, model, num_ftrs, num_classes):
        super(fully_connected, self).__init__()
        self.model = model
        self.fc_4 = nn.Linear(num_ftrs, num_classes)

    def forward(self, x):
        x = self.model(x)
        x = torch.flatten(x, 1)
        out_1 = x
        out_3 = self.fc_4(x)
        out_3 = torch.relu(out_3)
        return out_1, out_3

In [2]:
# prepare necessary arguments and WSI sample list

tif_list = '/data1/r20user3/shared_project/Hist2Cell/data/human_lung_cell2location/'
tif_list = glob(tif_list + '/W*[!xlsx|ipynb]')
tif_list.sort()
tif_list

['/data1/r20user3/shared_project/Hist2Cell/data/human_lung_cell2location/WSA_LngSP10193345',
 '/data1/r20user3/shared_project/Hist2Cell/data/human_lung_cell2location/WSA_LngSP10193346',
 '/data1/r20user3/shared_project/Hist2Cell/data/human_lung_cell2location/WSA_LngSP10193347',
 '/data1/r20user3/shared_project/Hist2Cell/data/human_lung_cell2location/WSA_LngSP10193348',
 '/data1/r20user3/shared_project/Hist2Cell/data/human_lung_cell2location/WSA_LngSP8759311',
 '/data1/r20user3/shared_project/Hist2Cell/data/human_lung_cell2location/WSA_LngSP8759312',
 '/data1/r20user3/shared_project/Hist2Cell/data/human_lung_cell2location/WSA_LngSP8759313',
 '/data1/r20user3/shared_project/Hist2Cell/data/human_lung_cell2location/WSA_LngSP9258463',
 '/data1/r20user3/shared_project/Hist2Cell/data/human_lung_cell2location/WSA_LngSP9258464',
 '/data1/r20user3/shared_project/Hist2Cell/data/human_lung_cell2location/WSA_LngSP9258467',
 '/data1/r20user3/shared_project/Hist2Cell/data/human_lung_cell2location/WSA

In [3]:
# define test slides list

test_slides = list()
for tif in tif_list:
    tif_path = tif.split('/')[-1]
    test_slides.append(tif_path)
test_slides

['WSA_LngSP10193345',
 'WSA_LngSP10193346',
 'WSA_LngSP10193347',
 'WSA_LngSP10193348',
 'WSA_LngSP8759311',
 'WSA_LngSP8759312',
 'WSA_LngSP8759313',
 'WSA_LngSP9258463',
 'WSA_LngSP9258464',
 'WSA_LngSP9258467',
 'WSA_LngSP9258468']

In [4]:
# prepare my GPU

gpu_list = [4]
gpu_list_str = ','.join(map(str, gpu_list))
os.environ.setdefault("CUDA_VISIBLE_DEVICES", gpu_list_str)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [5]:
# define test dataset transform

test_transform_pcam = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

In [6]:
import scanpy as sc

adata = sc.read("/data1/r20user3/shared_project/Hist2Cell/data/human_lung_cell2location/sp.X_norm5e4_log1p.h5ad")
center_col = adata.obs.array_col
center_row = adata.obs.array_row
center_col

spot_id
WSA_LngSP8759311_AAACAAGTATCTCCCA-1     102
WSA_LngSP8759311_AAACAGAGCGACTCCT-1      94
WSA_LngSP8759311_AAACATTTCCCGGATT-1      97
WSA_LngSP8759311_AAACCCGAACGAAATC-1     115
WSA_LngSP8759311_AAACCGTTCGTCCAGG-1      42
                                       ... 
WSA_LngSP10193348_TTGTGTATGCCACCAA-1     60
WSA_LngSP10193348_TTGTGTTTCCCGAAAG-1     59
WSA_LngSP10193348_TTGTTAGCAAATTCGA-1     42
WSA_LngSP10193348_TTGTTGTGTGTCAAGA-1     77
WSA_LngSP10193348_TTGTTTCACATCCAGG-1     42
Name: array_col, Length: 20770, dtype: int64

In [14]:
from tqdm import tqdm

batch_size = 512
patch_dir = "/data1/r20user3/shared_project/Hist2Cell/data/human_lung_cell2location"

graphs = dict()
for slide in tqdm(test_slides):
    print(slide)
    test_data = PTC_cell_2x(root=patch_dir, slide=slide,transform=test_transform_pcam)
    test_loader = torch.utils.data.DataLoader(test_data, batch_size=batch_size, shuffle=False, num_workers=4)

    spots_coord = pd.read_csv(os.path.join("/data1/r20user3/shared_project/Hist2Cell/data/human_lung_cell2location", slide, "2x_spots.csv"), index_col=0)

    with torch.no_grad():
        # model.eval()
        test_prediction_array = []
        test_label_array = []
        test_name_array = []

        for name, data, label in test_loader:
            test_name_array.append(list(name))
            data = data.to(device)
            label = label.to(device)
            label = label.float()
            label = label.squeeze()
            test_label_array.append(label.cpu().detach().numpy())
            
            # features, output = model(data)
            # test_prediction_array.append(features.squeeze().cpu().detach().numpy())
            test_prediction_array.append(data.cpu().detach().numpy())

    for i in range(len(test_prediction_array)):
        if len(test_prediction_array[i].shape) <= 1:
            test_prediction_array[i] = test_prediction_array[i][np.newaxis, :]
    for i in range(len(test_label_array)):
        if len(test_label_array[i].shape) <= 1:
            test_label_array[i] = test_label_array[i][np.newaxis, :]

    test_names = list()
    for names in test_name_array:
        test_names = test_names+names
    new_test_names = list()
    for name in test_names:
        if "WSA" in name:
            new_test_names.append(str(center_col.loc[name])+"x"+str(center_row.loc[name]))
        else:
            new_test_names.append(name)
    
    test_prediction_array = np.concatenate(test_prediction_array)
    test_label_array = np.concatenate(test_label_array)
    
    test_node_x_y = list()
    for item in new_test_names:
        test_node_x_y.append([float(item.split('x')[0]), float(item.split('x')[1])])

    adj = np.zeros((len(test_node_x_y), len(test_node_x_y)))

    for i in range(len(test_node_x_y)):
        for j in range(len(test_node_x_y)):
            if i == j:
                adj[i][j] = 1.0
            else:
                x1 = test_node_x_y[i][0]
                y1 = test_node_x_y[i][1]
                x2 = test_node_x_y[j][0]
                y2 = test_node_x_y[j][1]

                if x2 <= x1 - 2 or x2 >= x1 + 2 or y2 <= y1 - 1 or y2 >= y1 + 1:
                    continue
                else:
                    adj[i][j] = 1.0

    graphs[slide] = {
        'features': test_prediction_array,
        'labels': test_label_array,
        'adj': adj,
        'names': test_names,
        'coord': spots_coord.loc[test_names].values,
    }

    print("OK")

  0%|          | 0/11 [00:00<?, ?it/s]

WSA_LngSP10193345


  9%|▉         | 1/11 [22:24<3:44:02, 1344.22s/it]

OK
WSA_LngSP10193346


 18%|█▊        | 2/11 [31:50<2:12:59, 886.64s/it] 

...

OK
WSA_LngSP9258467


 91%|█████████ | 10/11 [46:23<02:18, 138.06s/it]

OK
WSA_LngSP9258468


100%|██████████| 11/11 [46:54<00:00, 255.83s/it]

OK


In [15]:
graphs['WSA_LngSP9258468']['features'].shape

(8642, 3, 224, 224)

In [16]:
graphs['WSA_LngSP9258468']['coord'].shape

(8642, 2)

In [17]:
graphs['WSA_LngSP9258468']['names']

['55.5x65.5',
 '60.5x62.5',
 '60.5x66.5',
 '104.5x48.5',
 '37x60',
 'WSA_LngSP9258468_GGACACAAGTTTACAC-1',
 '98.5x52.5',
 '10x13',
 '23.5x29.5',
 'WSA_LngSP9258468_CGCAAACACGAGTTAC-1',
 'WSA_LngSP9258468_CGTTGAGTAATTGCGT-1',
 '23x18',
 '17x12',
 '12.5x19.5',
 '22.5x14.5',
 'WSA_LngSP9258468_TACCAGGAATCCCGTC-1',
 '101x34',
 'WSA_LngSP9258468_ACACACCAGGACCAGT-1',
 '0.5x22.5',
 '66.5x50.5',
 '83.5x54.5',
 '32x25',
 'WSA_LngSP9258468_ACAGCGACATTCTCAT-1',
 'WSA_LngSP9258468_TAATTGGAATCGGGAA-1',
 '31x32',
 '18x37',
 '42.5x53.5',
 'WSA_LngSP9258468_CCAAATAACAAGATTC-1',
 '24x19',
 '103x30',
 '24x31',
 'WSA_LngSP9258468_CCCTCCTCGCTCGTAT-1',
 '65.5x49.5',
 '26.5x24.5',
 '87.5x63.5',
 '54.5x56.5',
 'WSA_LngSP9258468_GTGCCTAGCTATGCTT-1',
 '62x63',
 '13.5x11.5',
 '72.5x56.5',
 '59x50',
 'WSA_LngSP9258468_GTCGGGAAGCAGAAAC-1',
 '29.5x53.5',
 '98.5x56.5',
 'WSA_LngSP9258468_CCTTGTGAACGTGGTT-1',
 'WSA_LngSP9258468_GGAACTTTGGCGATTA-1',
 '47.5x39.5',
 '54.5x64.5',
 '110.5x54.5',
 '98.5x2.5',
 'WSA_LngSP9

In [18]:
graph = graphs
for slide in graph:
    print(slide)

WSA_LngSP10193345
WSA_LngSP10193346
WSA_LngSP10193347
WSA_LngSP10193348
WSA_LngSP8759311
WSA_LngSP8759312
WSA_LngSP8759313
WSA_LngSP9258463
WSA_LngSP9258464
WSA_LngSP9258467
WSA_LngSP9258468


In [19]:
from torch import Tensor
import torch
import os

for slide_name in graph:
    x = Tensor(graph[slide_name]['features'])
    from torch_geometric.utils import dense_to_sparse
    adj = Tensor(graph[slide_name]['adj'])
    edge_index, _ = dense_to_sparse(adj)
    y = Tensor(graph[slide_name]['labels'])
    from torch_geometric.data import Data
    pos = Tensor(graph[slide_name]['coord'])

    data = Data(x=x, edge_index=edge_index, y=y, pos=pos)
    
    torch.save(data, os.path.join("/data1/r20user3/shared_project/Hist2Cell/code/data_preprocessing/rawimg_graph/human_lung_cell2location_2x", slide_name+".pt"))

In [20]:
import torch

data1 = torch.load("/data1/r20user3/shared_project/Hist2Cell/code/data_preprocessing/rawimg_graph/human_lung_cell2location_2x/WSA_LngSP8759311.pt")
data2 = torch.load("/data1/r20user3/shared_project/Hist2Cell/code/data_preprocessing/rawimg_graph/human_lung_cell2location_2x/WSA_LngSP10193345.pt")

In [21]:
from torch_geometric.data import Batch

data = Batch.from_data_list([data1, data2])
data

DataBatch(x=[20925, 3, 224, 224], edge_index=[2, 221005], y=[20925, 330], pos=[20925, 2], batch=[20925], ptr=[3])

In [22]:
from random import shuffle
from torch_geometric.loader import NeighborLoader
import torch_geometric

torch_geometric.typing.WITH_PYG_LIB = False

from torch_geometric.seed import seed_everything
seed_everything(0)

loader = NeighborLoader(
    data,
    # Sample 30 neighbors for each node for 2 iterations
    num_neighbors=[-1, -1],
    # Use a batch size of 128 for sampling training nodes
    batch_size=1,
    directed=False,
    input_nodes=None,
    # disjoint=True,
    shuffle=True
)

i = 0
for sampled_data in loader:
    print(sampled_data.input_id)
    i = i + 1
    if i > 10:
        break

tensor([15501])
tensor([19648])
tensor([6086])
tensor([18263])
tensor([15119])
tensor([7066])
tensor([6999])
tensor([19100])
tensor([6320])
tensor([17509])
tensor([2401])


/home/r20user3/anaconda3/envs/gene/lib/python3.11/site-packages/torch_geometric/sampler/neighbor_sampler.py:50: UserWarning: Using '{self.__class__.__name__}' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  warnings.warn("Using '{self.__class__.__name__}' without a "


In [23]:
sampled_data

DataBatch(x=[33, 3, 224, 224], edge_index=[2, 265], y=[33, 330], pos=[33, 2], ptr=[3], n_id=[33], e_id=[265], input_id=[1], batch_size=1)

In [24]:
sampled_data.pos

tensor([[11985.5000,  9338.5000],
        [11821.5000,  9243.0000],
        [12095.0000,  9339.0000],
        [12150.0000,  9434.0000],
        [12040.5000,  9434.0000],
        [12150.0000,  9243.5000],
        [11876.0000,  9338.0000],
        [12040.5000,  9243.5000],
        [11821.0000,  9433.5000],
        [11931.0000,  9433.5000],
        [11931.0000,  9243.0000],
        [11657.0000,  9338.0000],
        [11767.0000,  9148.0000],
        [11657.0000,  9147.5000],
        [11986.0000,  9148.0000],
        [11876.5000,  9148.0000],
        [11766.5000,  9338.0000],
        [11712.0000,  9243.0000],
        [12205.0000,  9339.0000],
        [12260.0000,  9243.5000],
        [12260.0000,  9434.0000],
        [12095.5000,  9529.0000],
        [11986.0000,  9529.0000],
        [12205.0000,  9529.0000],
        [12315.0000,  9339.0000],
        [12314.5000,  9529.5000],
        [11876.0000,  9529.0000],
        [12315.0000,  9148.0000],
        [12095.5000,  9148.0000],
        [12205